# Áudio de consulta: síntese, transcrição e análise de texto

Este notebook mostra o caminho **áudio → texto → eventos**:

1. Os roteiros de consulta (`data/roteiros/`) viram áudio com o **Azure Text to Speech**.
2. Cada áudio é transcrito com o **Azure Speech to Text**.
3. A transcrição é analisada com o **Azure Text Analytics** (sentimento e frases-chave) e com uma lista própria de **termos críticos**.

> Os áudios de consulta são **sintéticos**: não há dataset público de consultas médicas em português. A análise de voz real (alterações vocais) está no notebook 02.

In [1]:
import os
import sys
from pathlib import Path

RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

import pandas as pd

from src.audio.sintese import sintetizar
from src.audio.texto import analisar_texto, consultar_azure, encontrar_termos_criticos
from src.audio.transcricao import transcrever

pd.set_option("display.max_colwidth", 120)

## 1. Roteiros

In [2]:
roteiros = {p.stem: p.read_text(encoding="utf-8").strip() for p in sorted(Path("data/roteiros").glob("*.txt"))}
for nome, texto in roteiros.items():
    print(f"[{nome}]")
    print(texto)
    print()

[consulta_01_rotina]
Bom dia, doutora. Estou me sentindo bem esta semana. Tenho dormido a noite inteira e voltei a fazer minhas caminhadas pela manhã. O apetite está normal e estou tomando os remédios nos horários certos. Não tive nenhum problema desde a última consulta.

[consulta_02_cansaco]
Boa tarde, doutor. Nos últimos dias tenho sentido muito cansaço. Qualquer atividade simples, como arrumar a cama, já me deixa com fadiga. Também estou com insônia, acordo várias vezes durante a noite. Fora isso, não sinto dor, mas estou sem disposição para nada.

[consulta_03_falta_de_ar]
Doutora, estou preocupado. Desde ontem sinto falta de ar, até quando estou parado. Hoje de manhã tive uma dor no peito muito forte, que durou alguns minutos. Também senti palpitação e precisei me sentar. Estou com medo de que seja algo grave.

[consulta_04_medicacao]
Doutor, depois que a dose do remédio foi alterada eu não estou bem. Tenho sentido tontura quando me levanto e um pouco de confusão, esqueço onde de

## 2. Síntese de fala (Azure Text to Speech)

Cada roteiro vira um arquivo `.wav` em `data/amostras/`. Usamos duas vozes neurais em português para variar os pacientes.

In [3]:
vozes = ["pt-BR-FranciscaNeural", "pt-BR-AntonioNeural"]
audios = {}
for i, (nome, texto) in enumerate(roteiros.items()):
    audios[nome] = sintetizar(texto, Path("data/amostras") / f"{nome}.wav", voz=vozes[i % 2])
    print(f"{audios[nome].as_posix()}  ({audios[nome].stat().st_size / 1024:.0f} KB, voz {vozes[i % 2]})")

data/amostras/consulta_01_rotina.wav  (481 KB, voz pt-BR-FranciscaNeural)


data/amostras/consulta_02_cansaco.wav  (656 KB, voz pt-BR-AntonioNeural)


data/amostras/consulta_03_falta_de_ar.wav  (489 KB, voz pt-BR-FranciscaNeural)


data/amostras/consulta_04_medicacao.wav  (559 KB, voz pt-BR-AntonioNeural)


## 3. Transcrição (Azure Speech to Text)

Para conferir a qualidade, comparamos cada transcrição com o roteiro original pela **taxa de palavras iguais**.

In [4]:
import difflib
import re


def palavras(texto):
    return re.findall("[a-zà-ú]+", texto.lower())


def taxa_de_acerto(original, transcrito):
    """Proporção de palavras do roteiro que aparecem, na ordem, na transcrição."""
    a, b = palavras(original), palavras(transcrito)
    iguais = sum(bloco.size for bloco in difflib.SequenceMatcher(None, a, b).get_matching_blocks())
    return iguais / max(len(a), 1)


transcricoes = {nome: transcrever(caminho) for nome, caminho in audios.items()}
for nome, texto in transcricoes.items():
    print(f"[{nome}] acerto: {taxa_de_acerto(roteiros[nome], texto):.0%}")
    print(texto)
    print()

[consulta_01_rotina] acerto: 100%
Bom dia, doutora. Estou me sentindo bem esta semana. Tenho dormido a noite inteira e voltei a fazer minhas caminhadas pela manhã. O apetite está normal e estou tomando os remédios nos horários certos. Não tive nenhum problema desde a última consulta.

[consulta_02_cansaco] acerto: 100%
Boa tarde, doutor. Nos últimos dias, tenho sentido muito cansaço. Qualquer atividade simples como arrumar a cama, já me deixa com fadiga. Também estou com insônia, acordo várias vezes durante a noite. Fora isso, não sinto dor, mas estou sem disposição para nada.

[consulta_03_falta_de_ar] acerto: 100%
Doutora, estou preocupado desde ontem, sinto falta de ar até quando estou parado. Hoje de manhã tive uma dor no peito muito forte, que durou alguns minutos. Também senti palpitação e precisei me sentar. Estou com medo de que seja algo grave.

[consulta_04_medicacao] acerto: 100%
Doutor, depois que a dose do remédio foi alterada, eu não estou bem. Tenho sentido tontura quand

## 4. Análise de texto

### 4.1 Azure Text Analytics: sentimento e frases-chave

In [5]:
linhas = []
for nome, texto in transcricoes.items():
    analise = consultar_azure(texto)
    linhas.append({
        "consulta": nome,
        "sentimento": analise.sentimento,
        "negativo": round(analise.pontuacao_negativa, 2),
        "frases-chave": ", ".join(analise.frases_chave[:8]),
    })
pd.DataFrame(linhas)

,consulta,sentimento,negativo,frases-chave
0,consulta_01_rotina,positive,0.01,"horários certos, Bom dia, remédios, última consulta, doutora, caminhadas, manhã, apetite"
1,consulta_02_cansaco,mixed,0.72,"últimos dias, cansaço, insônia, disposição, tarde, doutor, atividade, cama"
2,consulta_03_falta_de_ar,negative,1.00,"palpitação, Doutora, ontem, falta, ar, dor, peito, minutos"
3,consulta_04_medicacao,negative,0.99,"remédio, confusão, Doutor, dose, sentido, tontura, coisas, fraqueza"


### 4.2 Termos críticos

In [6]:
pd.DataFrame([
    {"consulta": nome, "termos críticos": ", ".join(encontrar_termos_criticos(texto)) or "(nenhum)"}
    for nome, texto in transcricoes.items()
])

,consulta,termos críticos
0,consulta_01_rotina,(nenhum)
1,consulta_02_cansaco,"cansaço, fadiga, insônia"
2,consulta_03_falta_de_ar,"falta de ar, dor no peito, palpitação"
3,consulta_04_medicacao,"tontura, confusão, fraqueza"


## 5. Eventos gerados

É isto que o módulo entrega para a etapa de fusão: uma lista de eventos por consulta, cada um com sua severidade (1 leve, 2 moderada, 3 grave).

In [7]:
linhas = []
for nome, texto in transcricoes.items():
    eventos = analisar_texto(texto)
    if not eventos:
        linhas.append({"consulta": nome, "tipo": "(sem eventos)", "severidade": 0, "descrição": ""})
    for e in eventos:
        linhas.append({"consulta": nome, "tipo": e.tipo, "severidade": e.severidade, "descrição": e.descricao})
pd.DataFrame(linhas)

,consulta,tipo,severidade,descrição
0,consulta_01_rotina,(sem eventos),0,
1,consulta_02_cansaco,termo_critico,1,"Paciente mencionou ""cansaço"" na consulta"
2,consulta_02_cansaco,termo_critico,1,"Paciente mencionou ""fadiga"" na consulta"
3,consulta_02_cansaco,termo_critico,1,"Paciente mencionou ""insônia"" na consulta"
4,consulta_02_cansaco,sentimento_negativo,1,Sentimento negativo na fala do paciente (72%)
5,consulta_03_falta_de_ar,termo_critico,3,"Paciente mencionou ""falta de ar"" na consulta"
6,consulta_03_falta_de_ar,termo_critico,3,"Paciente mencionou ""dor no peito"" na consulta"
7,consulta_03_falta_de_ar,termo_critico,2,"Paciente mencionou ""palpitação"" na consulta"
8,consulta_03_falta_de_ar,sentimento_negativo,1,Sentimento negativo na fala do paciente (100%)
9,consulta_04_medicacao,termo_critico,2,"Paciente mencionou ""tontura"" na consulta"
